# Hardware input-calibration plotter

Compares FC telemetry (MAVSDK IMU: `Angular Velocity FRD`, `Acceleration`) against the commanded body-rate + thrust profile sent by `record_input_calibration.py` on the real Pi hardware.

Modeled after `PX4_Gazebo/notebooks/plotter_input_calibration.ipynb`, but **hardware has no ground truth for input calibration** (unlike the SITL/Gazebo version) -- `record_input_calibration.py`'s own docstring notes achieved rates come from `FC_node.getLogData()` (real IMU/EKF telemetry), which is exactly what input calibration is meant to characterize against the COMMAND. So this notebook only has the SITL notebook's §2 ("Input transfer: commanded vs achieved" -- the headline section) with no §1/§3 GT-diagnostic sections, since those need Gazebo pose ground truth that doesn't exist here.

**Battery-voltage caveat (read before trusting any run):** `HW_HOVER_THROTTLE_NORM` is only valid in the ~22.4-24.0V battery range (see memory `project_hover_voltage_curve` / `project_hover_throttle_search_2026_07_09`). A run recorded outside that range will show corrupted/noisy correlations from a systematic climb or sink bias, not real rate-tracking noise -- this notebook prints the battery voltage prominently for exactly that reason.

In [ ]:
import os
import sys
import numpy as np
import matplotlib.pyplot as plt
from scipy.interpolate import interp1d

sys.path.insert(0, os.path.abspath(os.path.join('..', 'scripts')))
from analyze_input_calibration import per_run_metrics, mad_trimmed_mean, AXES

np.set_printoptions(precision=3, suppress=True)
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['font.size'] = 11
plt.rcParams['axes.grid'] = True
plt.rcParams['grid.alpha'] = 0.3

# Keep these in sync with record_input_calibration.py / hardware_landing.py --
# NOT auto-imported since that module has hardware-only imports (mavsdk FC,
# Controller) that may not be importable off the Pi.
# 2026-07-21: MASS_KG corrected 1.230 -> 1.204 (user re-weighed the airframe).
# THRUST_SLOPE_N_PER_UNIT is the drag-corrected, reliability-filtered value --
# APPLIED live on the Pi in both hardware_landing.py and
# record_input_calibration.py as of 2026-07-21. See §6/§7 below for the full
# derivation (naive model was biased +6.1% high on implied mass; a linear
# vertical-velocity drag term closes that to +1.1%; quadratic drag is worse,
# so linear is the right functional form).
MASS_KG = 1.204                  # rechecked 2026-07-21, supersedes the 1.230 kg parts-sum figure
HOVER_THROTTLE_NORM = 0.388      # confirmed, valid ~22.4-24.0V only, unchanged
THRUST_SLOPE_N_PER_UNIT = 30.38  # drag-corrected value, APPLIED live 2026-07-21 (was 30.7)
G = 9.81

In [ ]:
# Pick a run directory.
#  - Default: most recent run in Test_Data/Calibration/Input/ (RUN_INDEX = 0).
#  - Set RUN_INDEX = 1 for the previous run, 2 for the one before, ...
#  - Prints battery voltage + sample count for every candidate so a
#    corrupted/partial/low-battery run is visible before you pick it.
CAL_DIR = os.path.join('..', 'Test_Data', 'Calibration', 'Input')
RUN_INDEX = 0
N_SHOW = 10

runs = sorted(
    (d for d in os.listdir(CAL_DIR)
     if os.path.isdir(os.path.join(CAL_DIR, d))
     and os.path.exists(os.path.join(CAL_DIR, d, 'Ground_Truth.npy'))),  # skip non-run dirs (e.g. HoverThrottle/)
    key=lambda d: os.path.getmtime(os.path.join(CAL_DIR, d)),
    reverse=True,
)

print(f'{"idx":>3s}  {"run":32s}  {"n_cmd":>6s}  {"battery":>8s}')
for i, d in enumerate(runs[:N_SHOW]):
    gt = np.load(os.path.join(CAL_DIR, d, 'Ground_Truth.npy'), allow_pickle=True).item()
    cmd = np.array(gt.get('Command', []))
    batt = gt.get('Battery Voltage')
    batt_str = f'{batt:.2f}V' if batt else 'n/a'
    marker = '  <-- selected' if i == RUN_INDEX else ''
    print(f'{i:>3d}  {d:32s}  {len(cmd):>6d}  {batt_str:>8s}{marker}')

run_dir = os.path.join(CAL_DIR, runs[RUN_INDEX])
print(f'\nLoading: {run_dir}')

## §1 — Data preparation

Load telemetry (IMU angular velocity + acceleration, body-FRD) and the commanded rate/thrust profile, aligned to a common timeline.

In [ ]:
tel = np.load(os.path.join(run_dir, 'Telemetry_Data.npy'), allow_pickle=True).item()
gt = np.load(os.path.join(run_dir, 'Ground_Truth.npy'), allow_pickle=True).item()

battery_voltage = gt.get('Battery Voltage')
print(f"Battery voltage: {battery_voltage:.2f}V" if battery_voltage else 'Battery voltage: NOT LOGGED (older run)')
if battery_voltage is not None and not (22.4 <= battery_voltage <= 24.0):
    print('*** WARNING: battery voltage outside the confirmed ~22.4-24.0V HOVER_THROTTLE_NORM=0.388 plateau. ***')
    print('*** Correlations/gains below may be corrupted by a systematic climb/sink bias, not real rate noise. ***')

t_cmd = np.array(gt['Time'])
cmd = np.array(gt['Command'])
nm = min(len(t_cmd), len(cmd))
t_cmd, cmd = t_cmd[-nm:], cmd[-nm:]
w_u = cmd[:, :3]   # commanded rate (rad/s), body-FRD
B_T_u = cmd[:, 3]  # commanded thrust delta (N, excess-over-hover)

print(f'\nn_cmd_samples = {nm},  profile duration = {t_cmd[-1] - t_cmd[0]:.2f}s')
print(f'unique commands:\n{np.unique(np.round(cmd, 3), axis=0)}')

t_imu = np.array(tel['IMU Timestamp']) - gt['Start Time']
w_t = np.array([[a.forward_rad_s, a.right_rad_s, a.down_rad_s]
                for a in tel['Angular Velocity FRD']])
a_t = np.array([[a.forward_m_s2, a.right_m_s2, a.down_m_s2]
                for a in tel['Acceleration']])
mask = (t_imu >= t_cmd[0]) & (t_imu <= t_cmd[-1])
t_imu, w_t, a_t = t_imu[mask], w_t[mask], a_t[mask]
print(f'n_imu_samples (within command window) = {mask.sum()}')

## §2 — Command vs achieved: body angular rate

Rate-loop tracking check: how well does PX4's inner loop follow the commanded body-FRD rate sent by `record_input_calibration.py`? The measured trace (PX4 IMU gyro) should lag the command by the rate-loop response time; a flat/uncorrelated measured trace (as currently seen on pitch) indicates the commanded rate isn't producing a resolvable response above noise at this amplitude.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(12, 9), constrained_layout=True)
for i, (ax, name) in enumerate(zip(axes, ['x (roll)', 'y (pitch)', 'z (yaw)'])):
    ax.plot(t_imu, w_t[:, i], label=r'$w_{\mathrm{meas}}$ (PX4 IMU)')
    ax.plot(t_cmd, w_u[:, i], label=r'$w_{\mathrm{cmd}}$ (commanded)',
            linestyle='--', linewidth=2)
    ax.set(title=f'Body angular rate $w_{{{name}}}$ (body-FRD, rad/s) — cmd vs IMU',
           xlabel='t (s)', ylabel='rad/s')
    ax.legend()
plt.show()

## §3 — Command vs achieved: thrust (via IMU specific force)

Thrust acts along body −z, so `a_down` (accelerometer specific force) ≈ −Thrust/mass. Expected `a_down` from the commanded `B_T` uses the model `a_down = -F_hover/mass + B_T·SLOPE_true/(SLOPE_assumed·mass)` (see `project_hover_throttle_search_2026_07_09` memory for the derivation) -- overlaying measured vs this model-predicted trace is a direct check on whether `THRUST_SLOPE_N_PER_UNIT` is still a good fit for this run's battery state.

In [ ]:
a_down_meas = a_t[:, 2]
a_down_hover_baseline = a_down_meas[np.abs(np.interp(t_imu, t_cmd, B_T_u)) < 1e-6].mean() if (np.abs(B_T_u) < 1e-6).any() else -G

B_T_on_imu = np.interp(t_imu, t_cmd, B_T_u)
a_down_expected = a_down_hover_baseline + B_T_on_imu * THRUST_SLOPE_N_PER_UNIT / (42.3 * MASS_KG)

plt.figure(figsize=(12, 5), constrained_layout=True)
plt.plot(t_imu, a_down_meas, label=r'$a_{down}$ measured (IMU)')
plt.plot(t_imu, a_down_expected, label=r'$a_{down}$ expected (from commanded $B_T$ + derived slope)', linestyle='--')
plt.axhline(a_down_hover_baseline, color='grey', linestyle=':', alpha=0.6,
            label=f'hover baseline = {a_down_hover_baseline:.2f} m/s²')
plt.title('Body thrust (specific force, m/s²) — commanded vs achieved')
plt.xlabel('t (s)'); plt.ylabel('a_down (m/s²)')
plt.legend()
plt.show()

## §4 — Per-axis command→response quality (this run)

Reuses `analyze_input_calibration.py::per_run_metrics` (single source of truth -- no duplicated logic) for Pearson r, cross-correlation lag, and gain per axis.

In [ ]:
m = per_run_metrics(run_dir)
if m is None:
    print('Insufficient samples in this run for per_run_metrics (too short / aborted too early).')
else:
    print(f'{"axis":4s}  {"r":>6s}  {"lag_ms":>7s}  {"gain":>7s}')
    for ax in AXES:
        print(f'{ax:4s}  {m[ax]["r"]:6.3f}  {m[ax]["lag_ms"]:7.1f}  {m[ax]["gain"]:7.3f}')
    print()
    print('  r close to 1   -> FC closely tracks cmd (good)')
    print('  r much < 1     -> FC response dominated by disturbance rejection / noise')
    print('  lag_ms         -> rate-loop deadtime (MAVSDK transit + PX4 inner loop)')
    print('  gain ~= 1.0    -> FC matches cmd amplitude; << 1 = underdamped tracking')

## §5 — Aggregate across all valid-battery runs

Same MAD-trimmed aggregation as `analyze_input_calibration.py`, but filtered here to only runs whose logged battery voltage falls in the confirmed 22.4-24.0V plateau -- excludes the corrupted low-battery batch and any run predating battery-voltage logging (`batt=n/a`).

In [ ]:
VOLTAGE_MIN, VOLTAGE_MAX = 22.4, 24.0

valid_runs = []
for d in runs:
    gt_d = np.load(os.path.join(CAL_DIR, d, 'Ground_Truth.npy'), allow_pickle=True).item()
    v = gt_d.get('Battery Voltage')
    if v is not None and VOLTAGE_MIN <= v <= VOLTAGE_MAX:
        valid_runs.append((d, v))

print(f'{len(valid_runs)} valid-battery runs (of {len(runs)} total):')
for d, v in valid_runs:
    print(f'  {d}  ({v:.2f}V)')

all_metrics = {ax: dict(r=[], lag_ms=[], gain=[]) for ax in AXES}
for d, v in valid_runs:
    mm = per_run_metrics(os.path.join(CAL_DIR, d))
    if mm is None:
        continue
    for ax in AXES:
        for key in ('r', 'lag_ms', 'gain'):
            all_metrics[ax][key].append(mm[ax][key])

print(f'\n{"axis":4s}  {"r":>10s}  {"lag_ms":>11s}  {"gain":>10s}')
for ax in AXES:
    r_tm, r_n, _ = mad_trimmed_mean(all_metrics[ax]['r'])
    l_tm, l_n, _ = mad_trimmed_mean(all_metrics[ax]['lag_ms'])
    g_tm, g_n, _ = mad_trimmed_mean(all_metrics[ax]['gain'])
    print(f'{ax:4s}  {r_tm:6.3f} (n={r_n})  {l_tm:7.1f} (n={l_n})  {g_tm:7.3f} (n={g_n})')

## §6 — Hover-throttle sine-sweep (`find_hover_throttle.py`, `THROTTLE_MODE=sine`)

Separate tool from input calibration, but same underlying question: what throttle holds hover, and what's the throttle→thrust slope? `find_hover_throttle.py` commands a sinusoidal throttle sweep (zero body rate) and fits IMU `a_down` (specific force) against commanded throttle. Cross-referenced here for a consolidated calibration picture.

**Timing-alignment bug (2026-07-20, confirmed fixed):** the live script anchors `imu_rel` to `imu_ts_ref` (the IMU clock timestamp captured at the exact instant the sweep starts), not `imu_ts[0]` (FC-connection time, tens of seconds earlier) — the old code silently fit pre-sweep accelerometer noise against the sine profile, "producing wildly inconsistent/sign-flipping slope estimates" (script's own docstring). Verified live on the Pi 2026-07-21: fix is in place.

**But today's 7 sine-fits (post-fix) still show 8.5-41 N/unit scatter (mass-corrected to 1.204 kg).** Root cause is NOT excitation amplitude (`SINE_AMP=0.03`) — the script's own history shows list-mode (much larger throttle steps, 0.35-0.75 range) gave the *same* 6-45 N/unit inconsistency before sine mode was even introduced. The real driver, confirmed by resyncing 2 runs from raw `.npz` telemetry below: **an unmodeled net-climb trend** (open-loop sweep, zero altitude feedback — even ~0.001-0.003 throttle mis-centering relative to true hover compounds into meters of drift over the 20s window via `0.5·a·t²`), plus a genuinely weak signal-to-noise ratio (expected sine-driven signal ≈ raw accelerometer noise floor, R²=0.30-0.74).

In [ ]:
HOVER_DIR = os.path.join('..', 'Test_Data', 'HoverThrottle')

# Today's (2026-07-21) 7 printed sine-fit results from find_hover_throttle.py
# session log (21072026.txt) -- only 2 of these have raw .npz telemetry saved
# locally (the rest were "aborted"/"interrupted" before a full save, or predate
# this download); slope converted to N/unit with the corrected MASS_KG.
SINE_FITS_TODAY = [
    dict(slope=-14.74, hover=0.3987, batt=24.13),
    dict(slope=-11.36, hover=0.3853, batt=23.14),
    dict(slope=-12.25, hover=0.3873, batt=23.02),
    dict(slope=-10.24, hover=0.3989, batt=22.82),
    dict(slope=-6.88,  hover=0.3975, batt=22.29),
    dict(slope=-22.10, hover=0.3906, batt=24.44),
    dict(slope=-33.31, hover=0.3868, batt=23.89),
]
print(f'{"hover":>8s} {"slope(m/s2/thr)":>16s} {"N/unit":>8s} {"batt(V)":>8s}')
n_per_unit = []
for f in SINE_FITS_TODAY:
    npu = MASS_KG * abs(f['slope'])
    n_per_unit.append(npu)
    print(f'{f["hover"]:8.4f} {f["slope"]:16.2f} {npu:8.2f} {f["batt"]:8.2f}')

hovers = [f['hover'] for f in SINE_FITS_TODAY]
print(f'\nhover throttle: mean={np.mean(hovers):.4f}  median={np.median(hovers):.4f}')
print(f'slope N/unit:   mean={np.mean(n_per_unit):.2f}  median={np.median(n_per_unit):.2f}  '
      f'range=[{min(n_per_unit):.2f}, {max(n_per_unit):.2f}]')

In [ ]:
from scipy.signal import correlate

def resync_and_validate(fname):
    """Empirical cross-correlation resync (no trust in any single saved
    reference timestamp) + fit-quality validation: R^2, residual noise vs
    expected sine-driven signal, and a_down-vs-TIME trend (net climb/descent
    confound the naive throttle-only fit doesn't model)."""
    d = np.load(os.path.join(HOVER_DIR, fname), allow_pickle=True)
    imu_ts = np.array(d['IMU Timestamp'], dtype=float)
    a_down = np.array([a.down_m_s2 for a in d['Acceleration']])
    sine_time = np.array(d['sine_time'], dtype=float)
    sine_throttle = np.array(d['sine_throttle'], dtype=float)
    dur = sine_time[-1]
    thr_interp = interp1d(sine_time, sine_throttle, bounds_error=False, fill_value=np.nan)

    def r_at(off):
        rel = imu_ts - off
        mask = (rel >= 0) & (rel <= dur)
        if mask.sum() < 50:
            return -2, None, None
        thr = thr_interp(rel[mask]); a = a_down[mask]
        valid = ~np.isnan(thr)
        if valid.sum() < 50:
            return -2, None, None
        c, t = thr[valid], a[valid]
        c_c, t_c = c - c.mean(), t - t.mean()
        denom = np.std(c_c) * np.std(t_c)
        r = abs(np.mean(c_c * t_c) / denom) if denom > 1e-9 else 0.0
        return r, c, t

    lo, hi = imu_ts.min(), imu_ts.max() - dur
    best_off, best_r = None, -2
    for off in np.arange(lo, hi, 0.05):
        r, _, _ = r_at(off)
        if r > best_r: best_r, best_off = r, off
    for off in np.arange(best_off - 0.2, best_off + 0.2, 0.002):
        r, _, _ = r_at(off)
        if r > best_r: best_r, best_off = r, off

    _, c, t = r_at(best_off)
    A = np.polyfit(c, t, 1)
    slope, intercept = A[0], A[1]
    pred = slope * c + intercept
    r2 = 1 - np.sum((t - pred) ** 2) / np.sum((t - t.mean()) ** 2)
    resid_std = (t - pred).std()
    n_per_unit = MASS_KG * abs(slope)
    hover_est = (-G - intercept) / slope

    rel = imu_ts - best_off
    mask = (rel >= 0) & (rel <= dur)
    tt = rel[mask][~np.isnan(thr_interp(rel[mask]))]
    trend = np.polyfit(tt, t, 1)[0]

    print(f'{fname}')
    print(f'  best_offset={best_off:.3f}s  slope={slope:.2f} m/s2/thr -> {n_per_unit:.2f} N/unit  hover_est={hover_est:.4f}')
    print(f'  R2={r2:.3f}  resid_std={resid_std:.3f} m/s2  expected_sine_signal={abs(slope)*0.03:.3f} m/s2 (peak)')
    print(f'  a_down trend vs TIME = {trend:+.4f} m/s2/s  (nonzero -> net climb/descent confound, not throttle-driven)')
    print()
    return dict(slope=slope, n_per_unit=n_per_unit, hover=hover_est, r2=r2)

resync_results = []
for f in sorted(os.listdir(HOVER_DIR)):
    if 'complete' in f and f.endswith('.npz'):
        d = np.load(os.path.join(HOVER_DIR, f), allow_pickle=True)
        if 'sine_time' in d and len(d['sine_time']) > 0:
            resync_results.append(resync_and_validate(f))

### §6 summary — hover throttle + thrust slope, three independent methods

| quantity | value |
|---|---|
| Hover throttle (sine-fit, today) | 0.385-0.399, mean 0.392, median 0.391 -- tight, consistent with the live `HOVER_THROTTLE_NORM=0.388` |
| Thrust slope: sine-sweep median | ~14.75 N/unit -- **unreliable**, see confound analysis above (only n=2 raw datasets, detrending tested and only marginally helped) |
| Thrust slope: input-cal, naive model, r>=0.5-filtered, R^2-weighted | 30.7 * 0.7830/0.8306 -- implied mass +6.1% high |
| **Thrust slope: input-cal, drag-corrected, r>=0.5-filtered, R^2-weighted** | **30.38 N/unit -- APPLIED live 2026-07-21, see §7** |

The drag-corrected input-cal cross-check is the most methodologically sound of these (proper r>=0.5 reliability filtering restricted to the 19 thrust-reliable runs, cross-correlation lag alignment, AND a vertical-velocity drag term added to the point-mass model) and is now the live value on the Pi. It landed close to the original pre-session 30.7 -- most of the earlier apparent "gap" between methods turned out to be explained once drag was modeled correctly.

**Mass-gap thread: RESOLVED.** Both the 2026-07-20 and this session's thrust-gain regressions independently overestimated mass relative to the known 1.204 kg. Root cause: aerodynamic drag (linear in vertical velocity) missing from the naive `a=F/m` point-mass model. Adding a `k*v_z` term to the regression closes the gap from +6.1% to +1.1% (well within noise). Ground effect and battery-voltage/thermal effects were tested and ruled out. See `project_hardware_drone_mass` memory for the full investigation, including a self-corrected methodology bug along the way (an intermediate pass accidentally dropped the reliability filter, producing a misleadingly large +15-19% gap that did not survive re-checking -- always confirm `r>=0.5` filtering is applied before trusting any R^2-weighted aggregate here).

## §7 — Finalized calibration factors (APPLIED live on the Pi, 2026-07-21)

**Methodology:** `analyze_input_calibration.py::r2_weighted_mean()` -- weight each run's contribution by `r^2` (variance explained) instead of a hard `r>=0.5` cutoff, which has an inherent boundary problem (r=0.49 fully excluded, r=0.51 fully included -- unstable exactly where it matters, as happened to `wy`). For the thrust axis specifically, the r>=0.5 reliability filter is applied FIRST to select the 19 reliable runs, THEN a linear vertical-velocity drag term is added to the regression before R^2-weighting -- see §6 summary above for why (closes the mass-implied gap from +6.1% to +1.1%).

These are **correction factors**: gain = achieved/commanded, so multiplying an intended command by `1/gain` before sending it should make the achieved response match intent.

**Status: all of these are now live** in `hardware_landing.py` (`RATE_CORRECTION`, `HW_THRUST_SLOPE=30.38`) and `record_input_calibration.py` (`HW_THRUST_SLOPE=30.38`, plus `INPUT_CAL_STEP_HOLD_S=1.0` and `INPUT_CAL_RECENTER_HOLD_S=2.0` from the cutoff investigation). **None have been validated on an actual flight yet.**

In [ ]:
from analyze_input_calibration import r2_weighted_mean

CAL_DIR_ALL = os.path.join('..', 'Test_Data', 'Calibration', 'Input_Clean')
all_runs = sorted(d for d in os.listdir(CAL_DIR_ALL) if os.path.isdir(os.path.join(CAL_DIR_ALL, d)))

# Rate axes: r^2-weighted over ALL runs (per_run_metrics gain, no drag term needed --
# that confound is thrust/vertical-specific, not a rate-tracking issue).
final_metrics = {ax: dict(r=[], gain=[]) for ax in AXES}
for d in all_runs:
    mm = per_run_metrics(os.path.join(CAL_DIR_ALL, d))
    if mm is None:
        continue
    for ax in AXES:
        final_metrics[ax]['r'].append(mm[ax]['r'])
        final_metrics[ax]['gain'].append(mm[ax]['gain'])

print(f'{"axis":6s}  {"gain":>8s}  {"n_eff":>7s}  {"correction (1/gain)":>20s}')
final_gain = {}
for ax in ('wx', 'wy', 'wz'):
    g, n_eff = r2_weighted_mean(final_metrics[ax]['r'], final_metrics[ax]['gain'])
    final_gain[ax] = g
    print(f'{ax:6s}  {g:8.3f}  {n_eff:7.2f}  {1/g:20.3f}')

# Thrust: r>=0.5 reliability filter FIRST (restrict to the 19 thrust-reliable
# runs), THEN fit a_down = gain*B_T + k*v_z + c (linear drag term) per run,
# THEN R^2-weight across those 19 -- see §6 for why each step matters.
def thrust_drag_corrected_gain(run_dir):
    tel = np.load(f'{run_dir}/Telemetry_Data.npy', allow_pickle=True).item()
    gt = np.load(f'{run_dir}/Ground_Truth.npy', allow_pickle=True).item()
    t_cmd = np.array(gt['Time']); cmd_full = np.array(gt['Command'])
    if cmd_full.ndim != 2 or cmd_full.shape[1] < 4:
        return None
    B_T = cmd_full[:, 3]
    nm = min(len(t_cmd), len(B_T))
    t_cmd, B_T = t_cmd[-nm:], B_T[-nm:]
    imu_ts = np.array(tel['IMU Timestamp']) - gt['Start Time']
    a_down = np.array([a.down_m_s2 for a in tel['Acceleration']])
    odo_ts = np.array(tel['Odometry Timestamp']) - gt['Start Time']
    vz = np.array([v.z_m_s for v in tel['Velocity Body']])
    mask = (imu_ts >= t_cmd[0]) & (imu_ts <= t_cmd[-1])
    if mask.sum() < 100:
        return None
    imu_ts, a_down = imu_ts[mask], a_down[mask]
    fs = 200.0
    t_uniform = np.arange(t_cmd[0], t_cmd[-1], 1 / fs)
    B_T_u = interp1d(t_cmd, B_T, bounds_error=False, fill_value=0.0)(t_uniform)
    a_u = interp1d(imu_ts, a_down, bounds_error=False, fill_value=float(a_down.mean()))(t_uniform)
    vz_u = interp1d(odo_ts, vz, bounds_error=False, fill_value=0.0)(t_uniform)
    X = np.column_stack([B_T_u, vz_u, np.ones_like(B_T_u)])
    coef, *_ = np.linalg.lstsq(X, a_u, rcond=None)
    pred = X @ coef
    r2 = 1 - np.sum((a_u - pred) ** 2) / np.sum((a_u - a_u.mean()) ** 2)
    return coef[0], max(r2, 0.0)

thrust_gains, thrust_r2s = [], []
for d in all_runs:
    mm = per_run_metrics(os.path.join(CAL_DIR_ALL, d))
    if mm is None or mm['thrust']['r'] < 0.5:   # reliability filter FIRST
        continue
    res = thrust_drag_corrected_gain(os.path.join(CAL_DIR_ALL, d))
    if res is None:
        continue
    g, r2 = res
    thrust_gains.append(g); thrust_r2s.append(r2)

thrust_gains, thrust_r2s = np.array(thrust_gains), np.array(thrust_r2s)
final_gain['thrust'] = np.sum(thrust_r2s * thrust_gains) / np.sum(thrust_r2s)
print(f'thrust  {final_gain["thrust"]:8.3f}  {len(thrust_gains):>7d}  (n reliable runs, drag-corrected)')

DRONE_MASS_KG_FINAL = 1.204
EXPECTED_THRUST_GAIN = 1.0 / DRONE_MASS_KG_FINAL
OLD_SLOPE = 30.7
new_thrust_slope = OLD_SLOPE * final_gain['thrust'] / EXPECTED_THRUST_GAIN

print(f'\nFINALIZED (APPLIED live on the Pi, 2026-07-21):')
print(f'  M_correction (rate axes, wx/wy/wz) = diag({1/final_gain["wx"]:.3f}, {1/final_gain["wy"]:.3f}, {1/final_gain["wz"]:.3f})')
print(f'  HW_HOVER_THROTTLE_NORM = {HOVER_THROTTLE_NORM}')
print(f'  HW_THRUST_SLOPE_N_PER_UNIT = {new_thrust_slope:.2f}  (pre-session value: {OLD_SLOPE})')
print(f'\nAll of these are wired into hardware_landing.py / record_input_calibration.py')
print(f'on the Pi. NOT yet validated on an actual flight.')